## 1. READ JSON

In [416]:
import json
import pandas as pd

dir = r"C:\Users\User\Documents\catalogue_parser\src\full_results.json"

with open(dir, "r", encoding="utf-8") as f:
    data = json.load(f)

# extract products list
products = data["products"]

# convert to dataframe
df = pd.json_normalize(products)

### Drop false attributes

In [417]:
cols_drop =['materials.wood', 'materials.seat', 'materials.seating', 'materials.rope', 'other.fabric_color_count',
             'materials.cushions', 'materials.cushion',  'materials.steel','colors.top', 'colors.stripe', 'other.related_model',
             'other.sold_in_pieces', 'other.pillows', 'other','materials' ,'other.variant_model',
            'other.variant_reference', 'other.includes', 'other.additional_seats', 'other.foldable_arm']

df.drop(columns=cols_drop, inplace=True)

### Rename those with no issues

In [418]:
mapping = {
    "materials.frame": "Furniture Material",
    "materials.armrest": "Armrest material",
    "materials.table_top": "Top material",
    "colors.cushion": "Cushion color",
    "colors.frame": "Frame color",
    "other.backrest": "Backrest Material_2",
    "materials.sofa_frame": "Furniture Material_2",
    "other.table_function": "Table details_2",
    "materials.construction": "Bases Material",
    "colors.fabric": "Fabric color",
    "materials.armrest_legs": "Armrest and Legs Material",
    "materials.top": "Top material_2",
    "materials.assembly": "Assembled",
    "colors.rope": "Rope color",
    "colors": "Colors color",
    "other.assembly": "Assembled_2",
    "materials.fabric": "Fabric Material",
    "colors.textile": "Textile color",
    "materials.finish": "Finish Material",
    "materials.armrest_and_slats": "Armrest and slats Material",
    "materials.table": "Table details",
    "materials.backrest": "Backrest Material",
    "colors.steel": "Steel color",
    "colors.PP": "PP color",
    "materials.rattan": "Rattan color",
    "colors.rattan": "Rattan color_2",
    "other.available_colors": "Available_Colors other",
    "materials.properties": "Waterproof",
    "other.adjustable_table": "Table details_3",
    "colors.accent": "Accent color",
    "other.product_codes": "price_2",
    "colors.primary": "Primary color",
    "other.fire_pit_size": "Fire pit size",
    "other.frame_material": "Fabric Material_2",
    "colors.tabletop": "Top color",
    "other.cushion_thickness": "Cushion thickness",
    "colors.aluminum": "Aluminum color",
    "materials.chairs": "Chair Material",
    "colors.table_top": "Top color_2",
    "colors.seat": "Seat color",
    "other.table_type": "Top material_5",
    "colors.cushion_color": "Cushion color_3",
    "colors.frame_color": "Frame color_2",
    "colors.textile_color": "Textile color_2",
    "colors.table_color": "Table color_2",
    "colors.chair_color": "Chair color_8",
    "colors.table": "Table color",
    "materials.textile": "Textile Material",
    "colors.chair": "Chair color_9",
    "colors.chairs": "Chair color_10",
    "materials.table_frame": "Table Frame Material",
    "materials.chair_seat": "Double padded",
    "colors.chair_white": "Chair color_3",
    "colors.chair_grey": "Chair color_4",
    "colors.chair_green": "Chair color_5",
    "colors.chair_blue": "Chair color_6",
    "colors.chair_red": "Chair color_7",
    "colors.chair_options": "Chair color_2",
    "materials.surface": "Surface Material",
    "materials.tabletop": "Top material_3",
    "materials.legs": "Legs Material",
    "other.tube_thickness": "Tube thickness",
    "other.armrest": "Armrest material_2",
    "colors.secondary": "Secondary color",
    "other.aluminum_color": "Aluminum color_2",
    "other.features": "Features_4",
    "dimensions.sun_lounger": "dimensions_2",
    "dimensions.side_table": "dimensions_3",
    "colors.polywood": "Polywood color",
    "other.steel_color": "Steel color_2",
    "other.foldable": "Foldable",
    "colors.cushions": "Cushion color_2",
    "materials.primary": "Furniture Material_3",
    "materials.secondary": "Furniture Material_4",
    "other.height_levels": "height Levels",
    "other.standpole": "Standpole size",
    "other.lighting": "Lighting",
    "materials.feature": "Features",
    "other.feature": "Features_3",
    "materials.base": "Base and weight",
    "materials.features": "Features_2",
    "other.packing": "Pieces sold",
    "other.filling": "Filling",
    "other.weight": "Weight",
    "materials.roof": "Roof",
    "colors.net": "Net color",
    "other.shade_size": "Shade Size"
}

df.rename(columns=mapping, inplace=True)

### Combine secondary and primary colors

In [419]:
df[(df['Primary color'].notna()) & (df['Secondary color'].notna())][['model','Primary color','Secondary color']]

,model,Primary color,Secondary color
646,PP921A,Grey,White
686,MOD#YC-055,White,Grey
861,MOD-300-18PUSHUP,lime green,cyan blue
862,MOD#300-8-LED,cyan blue,beige


In [420]:
df["Primary color"] = df.apply(
    lambda row: (
        row["Primary color"] + ", " + row["Secondary color"]
        if pd.notna(row["Secondary color"]) and row["Secondary color"] != ""
        else row["Primary color"]
    ),
    axis=1
)


In [421]:
df[(df['Primary color'].notna()) & (df['Secondary color'].notna())][['model','Primary color','Secondary color']]

,model,Primary color,Secondary color
646,PP921A,"Grey, White",White
686,MOD#YC-055,"White, Grey",Grey
861,MOD-300-18PUSHUP,"lime green, cyan blue",cyan blue
862,MOD#300-8-LED,"cyan blue, beige",beige


In [422]:
df.drop(columns=['Secondary color'], inplace=True)
df.rename(columns={"Primary color": "Color"}, inplace=True)

### Same name to all same attributes


* chair color

In [423]:
cols = [c for c in df.columns if c.startswith("Chair color")]
cols

['Chair color_8',
 'Chair color_9',
 'Chair color_10',
 'Chair color_3',
 'Chair color_4',
 'Chair color_5',
 'Chair color_6',
 'Chair color_7',
 'Chair color_2']

In [424]:
df["Chair color"] = df[cols].bfill(axis=1).iloc[:, 0]

In [425]:
df.drop(columns=[c for c in cols if c != "Chair color"], inplace=True)


In [426]:
df[df['Frame color'].notna()][['model','Frame color']]

,model,Frame color
0,MOD#S28100,White
1,MOD#HT9350,Gray
2,MOD#SI014,White
3,MOD#G23-7203A,Gray
4,Nato,Dark gray/Black
...,...,...
759,MOD#1666,Silver
761,MOD#PP789,Gray
763,MOD#1600-S,Black
765,MOD#1519,Black


* Top material

In [427]:
cols = [c for c in df.columns if c.startswith("Top material")]
cols

['Top material', 'Top material_2', 'Top material_5', 'Top material_3']

In [428]:
df["Top material"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Top material"], inplace=True)


* Features

In [429]:
cols = [c for c in df.columns if c.startswith("Features")]
cols
df["Features"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Features"], inplace=True)


* Colors

In [430]:
cols = [c for c in df.columns if c.startswith("Cushion color")]
cols
df["Cushion color"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Cushion color"], inplace=True)

cols = [c for c in df.columns if c.startswith("Frame color")]
cols
df["Frame color"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Frame color"], inplace=True)


cols = [c for c in df.columns if c.startswith("Rattan color")]
cols
df["Rattan color"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Rattan color"], inplace=True)


In [431]:
cols = [c for c in df.columns if c.startswith("Furniture Material")]
cols
df["Furniture Material"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Furniture Material"], inplace=True)

In [432]:

cols = [c for c in df.columns if c.startswith("Table color")]
cols
df["Table color"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Table color"], inplace=True)


cols = [c for c in df.columns if c.startswith("Top color")]
cols
df["Top color"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Top color"], inplace=True)


In [433]:

cols = [c for c in df.columns if c.startswith("Fabric Material")]
cols
df["Fabric Material"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Fabric Material"], inplace=True)


cols = [c for c in df.columns if c.startswith("Aluminum color")]
cols
df["Aluminum color"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Aluminum color"], inplace=True)



In [434]:

cols = [c for c in df.columns if c.startswith("dimensions")]
cols
df["dimensions"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "dimensions"], inplace=True)


cols = [c for c in df.columns if c.startswith("Textile color")]
cols
df["Textile color"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Textile color"], inplace=True)



In [435]:

cols = [c for c in df.columns if c.startswith("Steel color")]
cols
df["Steel color"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Steel color"], inplace=True)


cols = [c for c in df.columns if c.startswith("Table details")]
cols
df["Table details"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Table details"], inplace=True)



In [436]:
cols = [c for c in df.columns if c.startswith("color")]
cols
df["color"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "color"], inplace=True)


In [437]:

cols = [c for c in df.columns if c.startswith("Armrest material")]
cols
df["Armrest material"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Armrest material"], inplace=True)


cols = [c for c in df.columns if c.startswith("price")]
cols
df["price"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "price"], inplace=True)


In [438]:

cols = [c for c in df.columns if c.startswith("Backrest Material")]
df["Backrest Material"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Backrest Material"], inplace=True)


cols = [c for c in df.columns if c.startswith("Assembled")]
cols
df["Assembled"] = df[cols].bfill(axis=1).iloc[:, 0]
df.drop(columns=[c for c in cols if c != "Assembled"], inplace=True)


### True and false = 1 & 0

In [439]:
cols= ['Double padded', 'Foldable', 'Assembled','Waterproof']

In [440]:
df[cols] = df[cols].notna().astype(int)

In [463]:
df.columns

Index(['page', 'model', 'price', 'dimensions', 'seater', 'Furniture Material',
       'Armrest material', 'Top material', 'Cushion color', 'Frame color',
       'Bases Material', 'Fabric color', 'Armrest and Legs Material',
       'Assembled', 'Rope color', 'Colors color', 'Fabric Material',
       'Textile color', 'Finish Material', 'Armrest and slats Material',
       'Table details', 'Backrest Material', 'Steel color', 'PP color',
       'Rattan color', 'Waterproof', 'Accent color', 'Color', 'other.type',
       'capacity', 'Fire pit size', 'Top color', 'Cushion thickness',
       'Aluminum color', 'Chair Material', 'Seat color', 'Table color',
       'Table Frame Material', 'Double padded', 'Surface Material',
       'other.description', 'Legs Material', 'Tube thickness',
       'Polywood color', 'Textile Material', 'other.table_dimensions',
       'Foldable', 'height Levels', 'Standpole size', 'Lighting', 'Features',
       'Base and weight', 'Pieces sold', 'Filling', 'Weight', 'R

### outlier columns

In [442]:
mask = df["other.product_id_numbers"].notna()

df.loc[mask, "price"] = (
    df.loc[mask, "price"].astype(str)
    + ", "
    + df.loc[mask, "other.product_id_numbers"].astype(str)
)

df.drop(columns=['other.product_id_numbers'], inplace=True)

In [445]:
df[df["materials.armrest_table_bench"].notna()].dropna(axis=1, how="all")

,page,model,price,dimensions,seater,Furniture Material,Frame color,Fabric color,Assembled,materials.armrest_table_bench,Waterproof,Double padded,Foldable
76,38,MOD#DJR5028,88006,"3 Seats: 170*76*82 Cm, 1 Seat: 54*76*82 Cm, Be...",Set 3 + 1 + 1 + Bench + Table,Aluminium frame,dark gray/black,gray,0,Polywood armrest and table top and bench,0,0,0


In [446]:
mask = df["materials.armrest_table_bench"].notna()

df.loc[mask, "Armrest Material"] = "polywood"

In [447]:
df.drop(columns=['materials.armrest_table_bench'], inplace=True )

In [462]:
# 'materials.armrest_and_table',
#        'materials.armrest_and_table'

df[df["Available_Colors other"].notna()].dropna(axis=1, how="all")

# mask = df["materials.armrest_and_table"].notna()

# df.loc[mask, "Armrest Material"] = "polywood"

df.drop(columns=['Available_Colors other'], inplace=True )
# df.loc[df["Available_Colors other"].notna(), "Color"] = df["Available_Colors other"]